# Advanced Machine Learning: Regularisation, Cross-Validation and Ensembles

**Course alignment:** Advanced Machine Learning  
**Type:** Original supplementary practice  
**Scenario:** Predicting a continuous operational performance indicator from engineered features

This notebook demonstrates train/test separation, feature engineering, Ridge/Lasso regularisation, cross-validation, Random Forest, gradient boosting, voting and stacking. Synthetic data are used so that the workflow is self-contained.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV, LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, VotingRegressor, StackingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(17)
X0, y0 = make_regression(
    n_samples=1400,n_features=12,n_informative=8,
    noise=22,random_state=17
)
X = pd.DataFrame(X0,columns=[f"x{i}" for i in range(X0.shape[1])])
# Add a nonlinear effect and an interaction as a feature-engineering exercise.
X["x0_squared"] = X["x0"]**2
X["x1_x2_interaction"] = X["x1"]*X["x2"]
y = y0 + 18*np.sin(X["x3"]) + 12*X["x1_x2_interaction"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2,random_state=42)

## 1. Why preprocessing belongs inside a pipeline

The scaler is fitted separately within each training fold. This prevents information from validation folds leaking into preprocessing statistics.

In [ ]:
ridge = Pipeline([
    ("scale",StandardScaler()),
    ("model",RidgeCV(alphas=[.01,.1,1,10,100],cv=5))
])
lasso = Pipeline([
    ("scale",StandardScaler()),
    ("model",LassoCV(cv=5,max_iter=20000,random_state=42))
])
forest = RandomForestRegressor(
    n_estimators=250,min_samples_leaf=2,max_features=.8,
    random_state=42,n_jobs=-1
)
boost = HistGradientBoostingRegressor(
    learning_rate=.07,max_iter=180,l2_regularization=1.0,random_state=42
)

estimators=[("ridge",ridge),("forest",forest),("boost",boost)]
voting=VotingRegressor(estimators=estimators)
stacking=StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV(alphas=[.1,1,10]),
    cv=5,n_jobs=-1
)

models={
    "RidgeCV":ridge,"LassoCV":lasso,"Random Forest":forest,
    "Histogram Gradient Boosting":boost,
    "Voting Regressor":voting,"Stacking Regressor":stacking
}

## 2. Cross-validation on training data

The same shuffled K-fold partitions are used to compare training-set generalisation estimates. Cross-validation is used for model selection; the final test set remains untouched until the final comparison.

In [ ]:
cv=KFold(n_splits=5,shuffle=True,random_state=42)
rows=[]
for name,model in models.items():
    scores=cross_validate(
        model,X_train,y_train,cv=cv,
        scoring={"mae":"neg_mean_absolute_error","r2":"r2"},
        n_jobs=-1,return_train_score=False
    )
    rows.append({
        "model":name,
        "CV_MAE_mean":-scores["test_mae"].mean(),
        "CV_MAE_sd":scores["test_mae"].std(),
        "CV_R2_mean":scores["test_r2"].mean()
    })
cv_results=pd.DataFrame(rows).sort_values("CV_MAE_mean")
display(cv_results.round(3))

## 3. Final held-out test evaluation

In [ ]:
test_rows=[]
fitted={}
for name,model in models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    fitted[name]=model
    test_rows.append({
        "model":name,
        "MAE":mean_absolute_error(y_test,pred),
        "RMSE":np.sqrt(mean_squared_error(y_test,pred)),
        "R2":r2_score(y_test,pred)
    })
test_results=pd.DataFrame(test_rows).sort_values("MAE")
display(test_results.round(3))

## 4. Inspect residuals and permutation importance

In [ ]:
best_name=test_results.iloc[0]["model"]
best_model=fitted[best_name]
best_pred=best_model.predict(X_test)
residuals=y_test-best_pred

fig,ax=plt.subplots(figsize=(7,4))
ax.scatter(best_pred,residuals,alpha=.4,s=14)
ax.axhline(0,linestyle="--")
ax.set(title=f"Residual plot: {best_name}",xlabel="Predicted value",ylabel="Actual - predicted")
plt.tight_layout()
plt.show()

perm=permutation_importance(
    best_model,X_test,y_test,n_repeats=8,random_state=42,
    scoring="neg_mean_absolute_error",n_jobs=-1
)
importance=pd.Series(perm.importances_mean,index=X_test.columns).sort_values()
importance.plot(kind="barh",figsize=(7,5),title="Permutation importance on held-out test set")
plt.tight_layout()
plt.show()

## Concepts from the lecture

- **Underfitting:** model is too simple to capture the pattern.
- **Overfitting:** model captures noise and generalises poorly.
- **Ridge (L2):** shrinks coefficients and can stabilise correlated predictors.
- **Lasso (L1):** can shrink some coefficients to zero.
- **Bagging:** fits models in parallel on resampled data to reduce variance.
- **Boosting:** fits learners sequentially to correct previous errors.
- **Voting:** combines predictions from several models.
- **Stacking:** trains a meta-model on cross-validated base-model predictions.

## Practice and limitations

1. Compare RidgeCV and LassoCV selected regularisation strengths.
2. Investigate how the number of folds changes computation and score variability.
3. Compare the ensemble to the strongest single model. Ensembles do not guarantee improvement.
4. Inspect whether any highly influential variables are synthetic artifacts.
5. Repeat with an appropriately licensed real dataset and document the train/validation/test protocol.

This is not an evaluation of an operational forecasting system.